## Práctica implementación y análisis de un sistema de recomendación basado en memoria
### Objetivo
Implementar un filtro colaborativo basado en **memoria** (se puede utilizar el modelo basado en usuarios o el de productos visto en clase).

### Instrucciones
Detallado en la práctica

### Entregable
El alumno o grupo debe entregar un notebook perfectamente documentado donde se responda a las tareas propuestas.

### Referencias
Movielens dataset https://grouplens.org/datasets/movielens/

### Autores 
 - Ángel Ortiz de Lejarazu Sánchez
 - Ruben Castañeda Matute


In [1]:
# --- Imports y configuración gráfica ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Literal, Tuple

# NO necesario si ya importado.
from __future__ import annotations

# NO necesario si ya importado.

from sklearn.metrics.pairwise import cosine_similarity    # métrica estándar en espacios latentes
from sklearn.decomposition import PCA                     # reducir a 2D para visualizar

#### 1. Implementación básica del filtro colaborativo. Utilizar el dataset de ratings de MovieLens de 100k ratings (recommended for education and development, small)

Usaremos un fltro colaborativo basado en usuarios, esto quiere decir que la estimación del rating de un usuario u a la pelicula i se basa en los ratings al la película i de otros usuarios similares a u.

Los FC basados en usuarios requieren: 
- Una matriz de utilidad (e.g. ratings) 
- Una definición de similitud entre usuarios <br>
Aplicamos Pearson <br>

![Pearson](pearson.png)

- Una definición de vecindad (subconjunto más próximo a un usuario) 
- Una regla para predecir el rating en base al conjunto de usuarios de la vecindad.


In [2]:
# Carga de DATOS
movies_path = Path('dataset') / 'ml-latest-small' / 'movies.csv'
ratings_path = Path('dataset') / 'ml-latest-small' / 'ratings.csv'

print('Ruta movies:', movies_path)
print('Ruta ratings:', ratings_path)

# Cargar los DataFrames (asumiendo encoding utf-8 estándar)
movies = pd.read_csv(movies_path)
ratings = pd.read_csv(ratings_path)

print('Cargados: movies ->', movies.shape, ' | ratings ->', ratings.shape)

# str  y rating float para cálculos
ratings = ratings.astype({"userId": str, "movieId": str, "rating": float})
movies  = movies.astype({"movieId": str})

movie_titles = dict(zip(movies["movieId"], movies["title"]))

Ruta movies: dataset\ml-latest-small\movies.csv
Ruta ratings: dataset\ml-latest-small\ratings.csv
Cargados: movies -> (9742, 3)  | ratings -> (100836, 4)


In [3]:
# -----------------------  MATRIZ DE UTILIDAD -----------------------
def build_utility(ratings: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Crea matriz R (users x items (Películas)) con ratings y devuelve (R, user_means (medía rating del usuario)).
    Requiere columnas: userId, movieId, rating de ratings DataFrame.
    """
    #Convertimos los usuarios a filas , las películas a columnas y los ratings a valores de la matriz
    R = ratings.pivot_table(index="userId", columns="movieId", values="rating", aggfunc="mean")
    #Media de ratings por usuario (ignorando NaN)
    user_means = R.mean(axis=1)  
    return R, user_means # Devuelve la matriz de utilidad y la media de ratings por usuario

In [4]:
# -----------------------  SIMILITUD (PEARSON) -----------------------
def pearson_sim(u_vec: pd.Series, v_vec: pd.Series, *, min_common: int = 2, shrinkage: float = 10.0) -> float:
    """
    Calcula cuán similar es un usuario respecto a otro, comparando sus gustos (ratings) 
    en películas que ambos han visto.
    
    Pearson centrado en medias de cada usuario, con soporte mínimo y shrinkage.
    shrinkage: N/(N+shrinkage) * corr  (why: estabiliza con pocos co-ítems)
    """
    # Encuentra las películas que ambos usuarios han calificado
    common = u_vec.notna() & v_vec.notna()
    
    # Cuenta cuántas películas tienen en común
    n = int(common.sum())
    if n < min_common:
        return 0.0 # No hay suficientes películas en común para calcular la similitud
    
    # Extrae los ratings de las películas comunes
    u = u_vec[common]
    v = v_vec[common]
    
    # Centra los ratings restando la media de cada usuario
    u_c = u - u.mean()
    v_c = v - v.mean()
    
    # Calcula la correlación de Pearson
    # Si sus ratings son idénticos, la correlación es 1.0 , -1.0 si son opuestos.
    # Fórmula: corr(u,v) = Σ((x_i - x̄)(y_i - ȳ)) / sqrt(Σ(x_i - x̄)² * Σ(y_i - ȳ)²)
    # num : rating del usuario 1 menos su media por rating del usuario 2 menos su media
    # den : raíz cuadrada de la suma de los cuadrados de las diferencias al cuadrado
    
    num = float((u_c * v_c).sum())
    den = float(np.sqrt((u_c**2).sum()) * np.sqrt((v_c**2).sum()))
    if den == 0.0:
        return 0.0
    corr = num / den
    
    #Aplica shrinkage (ponderación): si tienen pocas películas en común, 
    # reduce la confianza en la similitud. Con muchas películas, el peso es casi 1.
    weight = n / (n + shrinkage)
    return float(weight * corr)

In [5]:
# -----------------------  PREDICCIÓN  -----------------------
def predict_user_item(R: pd.DataFrame, user_means: pd.Series, user_id: str, item_id: str,
    *, k: int = 20, min_common: int = 2, shrinkage: float = 10.0,
    ) -> float | None:
    """
    Predice r_hat(u,i) con vecinos de usuario (solo los que han valorado i).
    Devuelve None si no hay ningún vecino válido.
    """
    # Si el usuario o película no existen → devuelve None
    if user_id not in R.index or item_id not in R.columns:
        return None

# ------------------------- VECINDAD -------------------------- 

    #Encuentra candidatos para vecinos: usuarios que han valorado esa película (exceptuando el usuario actual
    rated_mask = R[item_id].notna()
    neighbors = R.index[rated_mask & (R.index != user_id)]
    if neighbors.empty:
        return None
    
# ------------------------- REGLA PREDICCIÓN --------------- 

    u_vec = R.loc[user_id]
    sims = []
    for v in neighbors:
        #Calcula similitud de Pearson entre el usuario y cada vecino candidato
        s = pearson_sim(u_vec, R.loc[v], min_common=min_common, shrinkage=shrinkage)
        if s != 0.0:
            sims.append((v, s))

    if not sims:
        return None
    # Ordena por similitud (en valor absoluto) y toma los k vecinos más similares (k=20 por defecto)
    sims.sort(key=lambda t: abs(t[1]), reverse=True)
    # 20 vecinos más similares
    sims = sims[:k]

    # Regla de predicción (centro por medias)
    # num : Σ (r(v,i) - mean(v)) * sim(u,v)
    # den : Σ |sim(u,v)|
    # en el numerador solo se consideran los vecinos que han valorado i
    # en el denominador se suman los valores absolutos de las similitudes
    num = 0.0
    den = 0.0
    for v, s in sims:
        #rv_i : rating del vecino v en el ítem i
        rv_i = R.at[v, item_id]
        if np.isnan(rv_i):
            continue
        # rv_i - user_means[v] = desviación del vecino respecto a su media
        # user_means[v] : media de ratings del vecino v
        # multiplicada por la similitud s entre el usuario y el vecino
        num += (rv_i - user_means[v]) * s
        #den : suma de los valores absolutos de las similitudes
        den += abs(s)

    if den == 0.0:
        return float(user_means.get(user_id, np.nan))
    
    # Predicción final: media del usuario + ajuste por vecinos
    return float(user_means[user_id] + num / den)

In [6]:
# ----------------------- 4) TOP-N RECOMENDACIONES -----------------------
def recommend_user(
    R: pd.DataFrame,
    user_means: pd.Series,
    user_id: str,
    *,
    k: int = 20,
    n: int = 10,
    min_common: int = 2,
    shrinkage: float = 10.0,
) -> pd.DataFrame:
    """
    Recomienda Top-n ítems no valorados por el usuario ordenando por predicción.
    """
    if user_id not in R.index:
        return pd.DataFrame(columns=["movieId", "pred"])
    unrated = R.columns[R.loc[user_id].isna()]
    preds = []
    for iid in unrated:
        p = predict_user_item(R, user_means, user_id, iid, k=k, min_common=min_common, shrinkage=shrinkage)
        if p is not None and not np.isnan(p):
            preds.append((iid, p))
    out = pd.DataFrame(preds, columns=["movieId", "pred"]).sort_values("pred", ascending=False).head(n)
    return out.reset_index(drop=True)


In [7]:
# Muestra Top-10 para un usuario ID dado y una predicción puntual

# ratings originales (sin usuarios nuevos)
R, user_means = build_utility(ratings)

# Define el ID del usuario
user_id = "1"
if user_id in R.index:     # Verifica que el usuario existe en la matriz R

    # Genera Top-10 recomendaciones para ese usuario
    # n=10 : 10 películas recomendadas
    # k=50 : considera los 50 vecinos más similares
    # min_common=3 : mínimo 3 películas en común para calcular similitud
    # shrinkage=10.0 : parámetro para estabilizar con pocos co-ítems
    topN = recommend_user(R, user_means, user_id=user_id, n=10, k=50, min_common=3, shrinkage=10.0)
    
    
    print(f"Top-10 recomendaciones para userId={user_id}:")
    for i, row in topN.iterrows(): # Itera las recomendaciones
        mid = row['movieId'] # Extrae el ID de la película de esa recomendación
        title = movie_titles.get(mid, f"movieId={mid}") # Obtiene el título de la película usando su ID
        
        # Imprime: número, título, ID y predicción del rating (3 decimales)
        print(f"{i+1:2d}. {title} (id={mid}) — punt: {row['pred']:.3f}")

    # Predicción puntual para la primera recomendación (si existe)

else:
    print(f"Usuario {user_id} no encontrado en la matriz R.")


Top-10 recomendaciones para userId=1:
 1. Crow, The: Wicked Prayer (2005) (id=61818) — punt: 7.555
 2. Derailed (2002) (id=72424) — punt: 7.555
 3. Bloodsport: The Dark Kumite (1999) (id=145951) — punt: 7.555
 4. Ménage (Tenue de soirée) (1986) (id=2742) — punt: 7.486
 5. À nous la liberté (Freedom for Us) (1931) (id=5560) — punt: 7.486
 6. Fifty Shades Darker (2017) (id=168712) — punt: 7.463
 7. Anaconda: The Offspring (2008) (id=95796) — punt: 7.375
 8. Old Dogs (2009) (id=72696) — punt: 7.375
 9. Rust and Bone (De rouille et d'os) (2012) (id=97024) — punt: 7.145
10. Match Factory Girl, The (Tulitikkutehtaan tyttö) (1990) (id=40491) — punt: 7.121


#### 2. Añadir uno o varios usuarios que representen a los miembros del equipo de prácticas (con ratings a un subconjunto de las películas del dataset) y mostrar las 10 mejores recomendaciones que proporciona a cada usuario añadido.

In [8]:
# Normaliza tipos y búsqueda simple de movieId por título
ratings = ratings.astype({"userId": str, "movieId": str, "rating": float})
movies  = movies.astype({"movieId": str})
movie_titles = dict(zip(movies["movieId"], movies["title"]))

# --- 1) Búsqueda simple de movieId por título (sin desempates por popularidad) ---
def find_movie_id(title_query: str) -> str | None:
    t = title_query.strip()
    # Match exacto (case-insensitive)
    exact = movies.loc[movies["title"].str.lower() == t.lower(), "movieId"]
    if not exact.empty:
        return exact.iloc[0]
    # Primer contains (case-insensitive)
    contains = movies.loc[movies["title"].str.contains(t, case=False, na=False), "movieId"]
    if not contains.empty:
        return contains.iloc[0]
    # Intento sin año si venía en paréntesis
    base = t.split("(")[0].strip()
    contains2 = movies.loc[movies["title"].str.contains(base, case=False, na=False), "movieId"]
    return contains2.iloc[0] if not contains2.empty else None

In [9]:
# Nuevos usuarios , usamos pruntuaciones practica anterior.
user1_ratings = [
    ('Toy Story', 5),
    ('Star Wars: Episode IV - A New Hope', 5),
    ('Terminator 2: Judgment Day', 4),
    ('Pulp Fiction', 3),
    ('Forrest Gump', 2),
    ('Silence of the Lambs', 4),
]
user2_ratings = [
    ('Toy Story', 3),
    ('Forrest Gump', 5),
    ('Shawshank Redemption', 5),
    ('Star Wars: Episode IV - A New Hope', 2),
    ('Pretty Woman', 4),
    ('Braveheart', 4),
]

# Define IDs únicos que no existen
UID_ANGEL = "10001"
UID_RUBEN = "10002"

def build_new_users_df(uid: str, pairs: list[tuple[str, float]]) -> pd.DataFrame:
    # Construye DataFrame de ratings para los nuevos usuarios
    rows = []
    for title, r in pairs: # Itera sobre cada película y su rating
        mid = find_movie_id(title) # Busca el ID de la película

        if mid is None:  # Si no encuentra la película
            print(f"No se encontró '{title}' — omitido")
            continue
        # Si encuentra la película, agrega una fila al DataFrame
        rows.append({"userId": uid, "movieId": str(mid), "rating": float(r)})
    # Crea un DataFrame con las filas, con las columnas en orden
    return pd.DataFrame(rows, columns=["userId","movieId","rating"])

new_angel = build_new_users_df(UID_ANGEL, user1_ratings)
new_ruben = build_new_users_df(UID_RUBEN, user2_ratings)
new_users_df = pd.concat([new_angel, new_ruben], ignore_index=True)
print(f"Añadidos: Ángel={len(new_angel)} · Rubén={len(new_ruben)} · Total={len(new_users_df)}")


Añadidos: Ángel=6 · Rubén=6 · Total=12


In [10]:
# Agrega los nuevos usuarios al DataFrame original de ratings

# Selecciona solo las 3 columnas necesarias del DataFrame original
 # DataFrame con los nuevos usuarios:
ratings_ext = pd.concat([ratings[["userId","movieId","rating"]], new_users_df], ignore_index=True)

#  Convierte un DataFrame de ratings en:
# 1. Una matriz de utilidad R (usuarios × películas)
# 2. Una Serie con la media de ratings por usuario
def build_utility(ratings_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
     # Crea una tabla pivotada (transformación de fila a columna)
     # Los IDs de usuario van a las FILAS
     # Los IDs de película van a las COLUMNAS
     # Los ratings son los VALORES de las celdas
     # Si hay duplicados, toma la media
    R = ratings_df.pivot_table(index="userId", columns="movieId", values="rating", aggfunc="mean")

    # Calcula la media de ratings para cada usuario
    user_means = R.mean(axis=1)
    
    # Devuelve ambas estructuras
    return R, user_means

# Construye la matriz de utilidad usando TODOS los ratings (originales + nuevos)
R, user_means = build_utility(ratings_ext)


In [11]:
## Top-10 para Ángel y Rubén ---
for uid, name in [(UID_ANGEL, "Ángel"), (UID_RUBEN, "Rubén")]: # Itera sobre los dos usuarios nuevos
    print(f"\n Top-10 para {name} (userId={uid}):")
     
    # Verifica que el usuario existe en la matriz R
    if uid not in R.index:
        print("  (usuario no presente en R)")
        continue # Si no existe, salta
    
    # Genera las 10 mejores recomendaciones para ese usuario
    # Le pasamos los parámetros:
    # R : matriz de utilidad
    # user_means : medias de ratings por usuario
    # uid : ID del usuario
    # n=10 : número de recomendaciones a generar
    # k=50 : número de vecinos a considerar
    # min_common=3 : mínimo 3 películas en común para calcular similitud
    # shrinkage=10.0 : parámetro para estabilizar con pocos co-ítems
    topN = recommend_user(R, user_means, uid, n=10, k=50, min_common=3, shrinkage=10.0)
    
    
    # Si no hay recomendaciones
    if topN.empty:
        print("  (sin candidatos)")
        continue
    
    # Itera las recomendaciones y las muestra
    # i : número de fila (0, 1, 2, ..., 9)
    # row : fila con columnas ["movieId", "pred"]
    for i, row in topN.iterrows():
        
        mid = row["movieId"] # Extrae el ID de la película recomendada
        title = movie_titles.get(mid, f"movieId={mid}") # Obtiene el título
        # Imprime: número, título, ID y predicción del rating
        print(f"  {i+1:2d}. {title} (id={mid}) — punt: {row['pred']:.3f}")
    


 Top-10 para Ángel (userId=10001):
   1. Human Centipede, The (First Sequence) (2009) (id=77427) — punt: 7.240
   2. Secret Society (2002) (id=8632) — punt: 6.952
   3. Cincinnati Kid, The (1965) (id=8494) — punt: 6.863
   4. While the City Sleeps (1956) (id=8236) — punt: 6.863
   5. Hanging Up (2000) (id=3299) — punt: 6.775
   6. Fireproof (2008) (id=64114) — punt: 6.740
   7. The Gracefield Incident (2015) (id=173307) — punt: 6.594
   8. Sorrow (2015) (id=138186) — punt: 6.594
   9. Christmas Carol, A (1977) (id=107013) — punt: 6.594
  10. Satanic (2016) (id=160872) — punt: 6.594

 Top-10 para Rubén (userId=10002):
   1. Superhero Movie (2008) (id=59014) — punt: 7.030
   2. Crow, The: Wicked Prayer (2005) (id=61818) — punt: 7.022
   3. Derailed (2002) (id=72424) — punt: 7.022
   4. Bloodsport: The Dark Kumite (1999) (id=145951) — punt: 7.022
   5. Julien Donkey-Boy (1999) (id=2964) — punt: 6.920
   6. Fall (1997) (id=1574) — punt: 6.920
   7. Chopper Chicks in Zombietown (1989) (id=

### 3. ¿Qué películas resultan más parecidas a una dada, por ejemplo “Toy Story”, según las valoraciones de los usuarios?


Medimos similitud de patrón de valoraciones entre películas (p. ej., Pearson entre columnas de la matriz usuarios×películas, centrando por la media). Devolvemos las pelis con mayor correlación con Toy Story entre los usuarios que han valorado ambas.

- Construimos la columna de Toy Story (ratings por usuario) y, para cada otra peli, su columna.
- Nos quedamos con los usuarios en común (los que valoraron ambas).
<br> ** Ajustamos con min_common (soporte mínimo): **
<br><br> *min_common es un umbral de soporte: exige que dos películas tengan al menos N usuarios en común antes de aceptar su similitud. Se aplica antes del cálculo de Pearson; si no llegan al umbral, la similitud se descarta (0) para evitar falsos positivos por muestras muy pequeñas.*
<br><br>
- Centramos cada columna por su media : 
  Esto quiere decir calcular la desviación, por ejemplo una Película : ratings = [4, 4, 2] → media 3.33 → centrados = [+0.67, +0.67, −1.33]

  La media que restas es la de cada película (no global de todo el dataset, ni por usuario), y solo sobre los usuarios en común entre las dos pelis que se compara.

 - Calculamos su correlación de Pearson
 <br><br>
![Pearson](pearson.png)
<br>
    <br> Si sus ratings son idénticos, la correlación es 1.0 , -1.0 si son opuestos.
    <br> Fórmula: corr(u,v) = Σ((x_i - x̄)(y_i - ȳ)) / sqrt(Σ(x_i - x̄)² * Σ(y_i - ȳ)²)
    <br> num : rating centrado de la película a por rating centrado de la película b
    <br> den : raíz cuadrada de la suma de los cuadrados de las diferencias al cuadrado
 
-  Shrinkage 𝑛 /(𝑛+𝜆) : **Suaviza una correlación de Pearson cuando hay pocos usuarios en común.**
<br> sim = (n/(n+λ)) * corr → si n es pequeño, multiplica por un número < 1 y reduce la similitud; si n es grande, el factor ≈ 1 y no la toca.

- Salida: (similitud, n_common) donde similitud ya está penalizada por shrinkage.
- Ordenas por similitud y se obtiene las más parecidas.

<br><br>
**En resumen, no busca “mismas puntuaciones” ni “misma media”. Busca correlación de patrones: películas cuya forma de ser valoradas por los mismos usuarios se parece a la de Toy Story.**

<br>**1.-** Tomamos a los usuarios que han valorado ambas (Toy Story y otra).
<br>**2.-** Aplicamos min_common (mínimo de usuarios en común) 
<br>**3.-** Centramos cada conjunto de ratings por la media de cada película (para quitar nivel).
<br>**4.-** Calculamos Pearson: alta similitud ⇔ los usuarios que ponen por encima de la media a Toy Story también ponen por encima de la media a la otra (y viceversa para por debajo).
<br>**5.-** Aplicamnos shrinkage (penaliza correlaciones con poco soporte).



In [12]:
# ----------------------- SIMILITUD ITEM-ITEM (PEARSON) -----------------------
# Funciones helper para calcular similitudes entre películas
"""
def _find_movie_id_simple(title_query: str) -> str | None:
    
    Busca movieId por título (exacto > contains > sin año)
    
    t = title_query.strip()
    # Match exacto (case-insensitive)
    exact = movies.loc[movies["title"].str.lower() == t.lower(), "movieId"]
    if not exact.empty:
        return exact.iloc[0]
    # Primer contains (case-insensitive)
    contains = movies.loc[movies["title"].str.contains(t, case=False, na=False), "movieId"]
    if not contains.empty:
        return contains.iloc[0]
    # Intento sin año si venía en paréntesis
    base = t.split("(")[0].strip()
    contains2 = movies.loc[movies["title"].str.contains(base, case=False, na=False), "movieId"]
    return contains2.iloc[0] if not contains2.empty else None

# Usa la función find_movie_id si existe, sino crea una simple
_find_mid = find_movie_id if "find_movie_id" in globals() else _find_movie_id_simple
"""
def pearson_sim_items(R: pd.DataFrame, ia: str, ib: str, min_common: int = 5, shrinkage: float = 10.0) -> tuple[float, int]:
    """
    Calcula similitud Pearson entre dos películas (columnas de R).
    Retorna (similitud, número_usuarios_comunes)
    
    ia, ib : movieId de dos películas
    min_common : mínimo usuarios que deben haber visto ambas para calcular similitud
    shrinkage : parámetro para estabilizar con pocos co-ítems
    """
    # Valida que ambas películas existen en R
    if ia not in R.columns or ib not in R.columns:
        #sino similitud 0 y 0 usuarios comunes
        return 0.0, 0
    
    # Encuentra usuarios que han visto ambas películas
    # Usuarios con rating no NaN en ambas columnas (películas)
    # common es una Serie booleana de usuarios que han visto ambas películas
    common = R[ia].notna() & R[ib].notna()
    # Cuenta cuántos usuarios tienen en común
    n = int(common.sum())
    # Si no hay suficientes usuarios en común
    if n < min_common:
        # no hay suficientes usuarios en común para calcular similitud
        return 0.0, n
    
    
    # Extrae ratings comunes de ambas películas 
    # a es el vector de ratings de la película ia
    # b es el vector de ratings de la película ib
    # common es una máscara booleana para los usuarios que han visto ambas películas
    # r.loc : selecciona filas (usuarios) y columnas (películas)
    a = R.loc[common, ia]
    b = R.loc[common, ib]
    
    # Centra cada película restando su media
    # como lo centra? 
    # centrar por la media de la película (item-based) 
    # quita el nivel medio de puntuación de la película y deja el patrón relativo por usuario
    # a y b son Series con ratings centrados
    a_c = a - a.mean()
    b_c = b - b.mean()
    
    # Calcula correlación de Pearson
    # Cálculo de Pearson:
    # a_c : rating centrado de la película a
    # b_c : rating centrado de la película b
    # num = Σ((a_c) * (b_c))
    # den = sqrt(Σ(a_c^2) * Σ(b_c^2))
    # Si den == 0.0 (caso raro: todos los valores iguales para una peli) → similitud 0 (no definida).
    # corr = num / den: valor entre -1 y 1.
    num = float((a_c * b_c).sum())
    den = float(np.sqrt((a_c**2).sum()) * np.sqrt((b_c**2).sum()))
    if den == 0.0:
        return 0.0, n
    corr = num / den
    
    # Aplica shrinkage: reduce peso si pocos usuarios en común
    w = n / (n + shrinkage)
    
    #Salida: (similitud, n_common) donde similitud ya está penalizada por shrinkage.
    return float(w * corr), n

def top_similar_items(R: pd.DataFrame, target_mid: str, k: int = 10, min_common: int = 5, shrinkage: float = 10.0) -> pd.DataFrame:
    """
    R es la matriz de utilidad (users x items).
    target_mid es el movieId de la película objetivo.
    k es el número de películas similares a encontrar.
    min_common es el mínimo de usuarios comunes para calcular similitud.
    shrinkage es el parámetro para estabilizar con pocos co-ítems.
    --------------------------------
    Encuentra las k películas más similares a target_mid según Pearson item-item.
    Retorna DataFrame con columnas: movieId, title, sim, n_common
    """
    sims = []
    # Itera sobre todas las películas
    # mid : movieId de cada película en R
    for mid in R.columns:
        if mid == target_mid:
            # Omite la misma película
            continue
        # Calcula similitud Pearson entre target_mid y mid
        # s : similitud
        # n : número de usuarios comunes
        s, n = pearson_sim_items(R, target_mid, mid, min_common=min_common, shrinkage=shrinkage)
        # Si la similitud no es 0 ni NaN, la añade a la lista
        if s != 0.0 and not np.isnan(s):
            # Añade tupla (movieId, similitud, n_common)
            sims.append((mid, s, n))
    
    # Crea DataFrame y ordena por similitud (desc) y luego por n_common (desc)
    # las ordena primero por similitud y luego por número de usuarios comunes
    # head(k) : toma las k más similares
    out = (pd.DataFrame(sims, columns=["movieId", "sim", "n_common"])
           .sort_values(["sim", "n_common"], ascending=[False, False])
           .head(k).reset_index(drop=True))
    
    # Añade títulos
    # mapea movieId a título usando movie_titles dict
    out["title"] = out["movieId"].map(movie_titles).fillna(out["movieId"])
    return out[["movieId", "title", "sim", "n_common"]]

In [13]:
# ============================================================================
# EJEMPLO: Top-10 películas similares a "Toy Story"
# ============================================================================

target = "Toy Story"
mid = find_movie_id(target)

if mid is None:
    print(f" No se encontró película que coincida con: '{target}'")
else:
    # target_title : título real de la película (por si hubo búsqueda por contains)
    # movie_titles.get(mid, target) : obtiene el título usando movieId, si no existe usa target
    target_title = movie_titles.get(mid, target)
    print(f"\n Top-10 películas similares a '{target_title}' (id={mid})")
    print(f"   (Pearson item-item | min de usuarios en común=5 | shrinkage=10.0)\n")
    
    # Calcula las 10 películas más similares a mid
    # k=10 : top 10 similares
    # min_common=5 : mínimo 5 usuarios comunes para calcular similitud
    # shrinkage=10.0 : parámetro para estabilizar con pocos co-ítems
    topK = top_similar_items(R, mid, k=10, min_common=5, shrinkage=10.0)
    
    if topK.empty:
        print("   (sin películas similares encontradas)")
    else:
        for i, r in topK.iterrows():
            print(f"   {i+1:2d}. {r['title']:50s}  similitud: {r['sim']:6.3f} | usuarios en comun: {int(r['n_common']):4d}")


 Top-10 películas similares a 'Toy Story (1995)' (id=1)
   (Pearson item-item | min de usuarios en común=5 | shrinkage=10.0)

    1. Toy Story 2 (1999)                                  similitud:  0.622 | usuarios en comun:   81
    2. Incredibles, The (2004)                             similitud:  0.569 | usuarios en comun:   77
    3. Aladdin (1992)                                      similitud:  0.560 | usuarios en comun:  107
    4. Finding Nemo (2003)                                 similitud:  0.555 | usuarios en comun:   87
    5. Darjeeling Limited, The (2007)                      similitud:  0.507 | usuarios en comun:   14
    6. Arachnophobia (1990)                                similitud:  0.489 | usuarios en comun:   30
    7. Erin Brockovich (2000)                              similitud:  0.478 | usuarios en comun:   40
    8. Wallace & Gromit: The Wrong Trousers (1993)         similitud:  0.472 | usuarios en comun:   40
    9. Toy Story 3 (2010)                        